# Project 10 — Attention From Scratch (QKV, masks, MHA, positions)

```
tokens(n x d) -> Q=XWq K=XWk V=XWv -> scores=QK^T/sqrt(d) -> +mask -> softmax -> out=AV
single-head -> multi-head (split d into h x dh, concat, proj Wo) -> + pos-encoding -> block
```

**Goal:** derive attention with your own hands — scaled dot-product, why `1/sqrt(d)`,
what a causal mask does, how MHA splits/concatenates heads, and what positional
encodings add. Verify every piece against `torch.nn.functional` and visualize maps.

**Learning objectives:** (1) write softmax+attention numerically stable from scratch;
(2) prove causal masking blocks future leakage with a unit test; (3) build MHA and
count its params; (4) compare sinusoidal vs learned positions; (5) read an attention map;
(6) measure O(n^2) scaling and CPU vs GPU throughput.


In [ ]:
import os # need os to create results/ and inspect working directory
import random # need random to seed the stdlib RNG for reproducibility
import time # need time for median-of-N micro-benchmarks
import csv # need csv to write results tables portably
import math # need math for sqrt/exp/sin/cos in hand-rolled ops
import numpy as np # need numpy for the from-scratch reference implementation
import torch # need torch as the framework reference (and GPU path if present)
import torch.nn.functional as F # need F for reference softmax/scaled_dot_product_attention
import matplotlib # need matplotlib for heatmaps and scaling plots
matplotlib.use('Agg') # force Agg backend so plots render headlessly
import matplotlib.pyplot as plt # need pyplot for heatmap + line-plot figures
SEED = 0 # fixed seed: every rerun gives identical numbers
random.seed(SEED) # seed stdlib RNG
np.random.seed(SEED) # seed numpy RNG
torch.manual_seed(SEED) # seed torch RNG (CPU determinism)
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and PNGs
DEV = 'cuda' if torch.cuda.is_available() else 'cpu' # pick GPU when present, else CPU
print(f'device={DEV} torch={torch.__version__}') # print env line for the README record


In [ ]:
# ---- single-head scaled dot-product attention, NumPy reference vs torch ----
def softmax_stable(x, axis=-1): # define numerically stable softmax along one axis
    m = x.max(axis=axis, keepdims=True) # row/col max: shifting keeps exp() in range
    e = np.exp(x - m) # subtract max then exp: max input to exp is 0, no overflow
    return e / e.sum(axis=axis, keepdims=True) # normalize so each row sums to 1.0
def attn_numpy(X, Wq, Wk, Wv, mask=None): # hand attention: X(n,d)->out(n,dv), mask adds -inf
    Q = X @ Wq # project inputs to queries (n,dk)
    K = X @ Wk # project inputs to keys (n,dk)
    V = X @ Wv # project inputs to values (n,dv)
    dk = Q.shape[1] # head dim: scale factor is sqrt(dk)
    s = (Q @ K.T) / math.sqrt(dk) # raw scores (n,n): dot similarity scaled down
    if mask is not None: # causal/pad mask path
        s = np.where(mask, s, -1e9) # disallowed slots -> -1e9 so softmax gives ~0
    w = softmax_stable(s, axis=-1) # attention weights: rows sum to 1
    return w @ V, w # mix values by weights; also return weights for viz/tests
def causal_mask(n): # build lower-triangular boolean mask incl. diagonal
    return np.tril(np.ones((n, n), dtype=bool)) # True = allowed (past+self), False = future
n, d, dk, dv = 8, 16, 8, 8 # toy sizes: small enough to eyeball, big enough to test
rng = np.random.default_rng(SEED) # seeded RNG: reproducible toy weights
X = rng.normal(size=(n, d)).astype(np.float64) # toy inputs (float64: strict reference)
Wq = rng.normal(size=(d, dk)).astype(np.float64) # toy query proj
Wk = rng.normal(size=(d, dk)).astype(np.float64) # toy key proj
Wv = rng.normal(size=(d, dv)).astype(np.float64) # toy value proj
out_np, w_np = attn_numpy(X, Wq, Wk, Wv, mask=causal_mask(n)) # run hand attention w/ causal mask
o_t = F.scaled_dot_product_attention(torch.tensor(X @ Wq).unsqueeze(0), torch.tensor(X @ Wk).unsqueeze(0), torch.tensor(X @ Wv).unsqueeze(0), is_causal=True)[0].numpy() # torch fused SDPA causal reference
print(f'max|hand-torch|={np.abs(out_np - o_t).max():.2e}') # must be ~1e-12 or the hand code is wrong
assert np.abs(out_np - o_t).max() < 1e-9, 'hand attention mismatches torch' # hard gate: fail loudly
leak = w_np[np.triu_indices(n, k=1)].max() # largest weight on any future position (must be ~0)
print(f'causal leak (max future weight)={leak:.2e}') # expect 0.0: mask blocks the future
assert leak < 1e-6, 'causal mask leaks future info' # hard gate on masking correctness
row = w_np.sum(axis=1) # each attention row must sum to 1
print(f'rowsum min/max={row.min():.6f}/{row.max():.6f}') # expect 1.000000/1.000000
# ---- throughput bench: numpy-CPU vs torch-CPU vs torch-GPU(auto-skip) ----
def bench(fn, reps=30): # median-of-reps timer: median resists outlier pauses
    for _ in range(5): # warmup runs: caches hot, GPU clocks up
        fn() # throwaway call
    ts = [] # collect per-rep times
    for _ in range(reps): # timed reps
        t0 = time.perf_counter() # start timer
        fn() # run once
        ts.append(time.perf_counter() - t0) # store elapsed
    return float(np.median(ts)) # median: robust center of timings
N2, D2, HD = 256, 64, 64 # bench sizes: n=256 shows O(n^2) without being slow
Xb = np.random.default_rng(1).normal(size=(N2, D2)).astype(np.float32) # bench inputs
Wqb = np.random.default_rng(2).normal(size=(D2, HD)).astype(np.float32) # bench Wq
Wkb = np.random.default_rng(3).normal(size=(D2, HD)).astype(np.float32) # bench Wk
Wvb = np.random.default_rng(4).normal(size=(D2, HD)).astype(np.float32) # bench Wv
t_np = bench(lambda: attn_numpy(Xb, Wqb, Wkb, Wvb)) # time hand numpy path (no mask)
Xc = torch.from_numpy(Xb) # torch copy of inputs (shares memory, zero-copy)
Qc, Kc, Vc = torch.from_numpy(Xb @ Wqb), torch.from_numpy(Xb @ Wkb), torch.from_numpy(Xb @ Wvb) # precomputed QKV: isolates attention kernel from proj
t_tc = bench(lambda: F.scaled_dot_product_attention(Qc.unsqueeze(0), Kc.unsqueeze(0), Vc.unsqueeze(0))) # time torch CPU SDPA
row_gpu = ('torch-gpu', '', 'skipped-no-gpu') # default GPU row: skipped
if torch.cuda.is_available(): # GPU branch: only when CUDA is present
    Qg = Qc.cuda().unsqueeze(0) # move Q to GPU once (outside timer)
    Kg = Kc.cuda().unsqueeze(0) # move K to GPU once
    Vg = Vc.cuda().unsqueeze(0) # move V to GPU once
    torch.cuda.synchronize() # drain GPU queue before timing
    t0 = time.perf_counter() # start (wall clock; sync below makes it honest)
    for _ in range(30): # GPU reps
        F.scaled_dot_product_attention(Qg, Kg, Vg) # fused GPU kernel
    torch.cuda.synchronize() # wait for all kernels before stopping clock
    row_gpu = ('torch-gpu', f'{(time.perf_counter()-t0)/30*1e3:.3f}', 'fused-sdpa') # fill GPU row
flops = 2 * N2 * N2 * HD # approx flops of QK^T + AV (2 matmuls, fused softmax ignored)
print(f'numpy {t_np*1e3:.2f}ms | torch-cpu {t_tc*1e3:.2f}ms | gpu {row_gpu}') # console table
print(f'approx attn flops={flops/1e6:.2f} MFLOP -> numpy {flops/t_np/1e9:.2f} GFLOP/s') # intensity note: O(n^2*d) flops on O(n^2) bytes
with open('./results/attention_bench.csv', 'w', newline='') as f: # save bench CSV for README
    w = csv.writer(f) # csv writer handle
    w.writerow(['backend', 'ms_per_call', 'note']) # header row
    w.writerow(['numpy-cpu', f'{t_np*1e3:.3f}', f'n={N2} d={D2}']) # hand numpy row
    w.writerow(['torch-cpu', f'{t_tc*1e3:.3f}', 'fused-sdpa']) # torch CPU row
    w.writerow(list(row_gpu)) # GPU row (or skip marker)


In [ ]:
# ---- multi-head attention + positional encodings, all verified ----
class MHA(torch.nn.Module): # minimal multi-head attention module (pre-norm style agnostic)
    def __init__(self, d, h): # build MHA: d model dim, h heads
        super().__init__() # required nn.Module init
        assert d % h == 0, 'd must split evenly over heads' # guard: dh must be integer
        self.h, self.dh = h, d // h # store head count and per-head dim
        self.Wq = torch.nn.Linear(d, d, bias=False) # fused Q proj for all heads
        self.Wk = torch.nn.Linear(d, d, bias=False) # fused K proj for all heads
        self.Wv = torch.nn.Linear(d, d, bias=False) # fused V proj for all heads
        self.Wo = torch.nn.Linear(d, d, bias=False) # output proj mixes heads back to d
    def forward(self, x, causal=True): # x: (n,d) -> (out(n,d), weights(h,n,n))
        n, d = x.shape # read seq len and model dim
        Q = self.Wq(x).view(n, self.h, self.dh).transpose(0, 1) # (h,n,dh): split d into h heads
        K = self.Wk(x).view(n, self.h, self.dh).transpose(0, 1) # (h,n,dh): same split for keys
        V = self.Wv(x).view(n, self.h, self.dh).transpose(0, 1) # (h,n,dh): same split for values
        s = (Q @ K.transpose(-1, -2)) / math.sqrt(self.dh) # (h,n,n): per-head scaled scores
        if causal: # causal branch: mask future
            m = torch.tril(torch.ones(n, n, dtype=torch.bool)) # lower-triangular allow mask
            s = s.masked_fill(~m, -1e9) # forbid future slots
        w = F.softmax(s, dim=-1) # (h,n,n): per-head weights
        o = (w @ V).transpose(0, 1).reshape(n, d) # (n,d): concat heads then...
        return self.Wo(o), w # ...mix through Wo; return weights for viz
torch.manual_seed(SEED) # reseed: same MHA weights every run
mha = MHA(d=32, h=4) # toy MHA: 4 heads x dh=8
xm = torch.randn(12, 32) # toy sequence: 12 tokens x 32 dim
om, wm = mha(xm) # forward with causal mask
print(f'MHA out {tuple(om.shape)} weights {tuple(wm.shape)}') # expect (12,32) and (4,12,12)
nparams = sum(p.numel() for p in mha.parameters()) # count params: 4*d*d (Q,K,V,O)
print(f'MHA params={nparams} (expect {4*32*32})') # 4096: attention is 4 square matrices
assert nparams == 4 * 32 * 32 # hard gate on param accounting (used again in P12)
def sinusoid(n, d): # sinusoidal positions: fixed, no params, generalizes to longer n
    pe = np.zeros((n, d)) # output table (n,d)
    pos = np.arange(n)[:, None] # (n,1): token positions
    div = np.exp(np.arange(0, d, 2) * -(math.log(10000.0) / d)) # geometric wavelengths 2pi..2pi*10000
    pe[:, 0::2] = np.sin(pos * div) # sin on even dims
    pe[:, 1::2] = np.cos(pos * div[:pe[:, 1::2].shape[1]]) # cos on odd dims (slice guards odd d)
    return pe # (n,d): nearby rows align more than far rows -> relative info for free
pe = sinusoid(16, 32) # build 16x32 table
print(f'sinusoid self-sim={float(pe[3] @ pe[3]):.2f} dist-sim={float(pe[3] @ pe[10]):.2f}') # nearby rows align more than far rows
emb_learned = torch.nn.Embedding(16, 32) # learned alternative: 16*32 params, capped at trained length
print(f'learned-pos params={sum(p.numel() for p in emb_learned.parameters())} vs sinusoid params=0') # tradeoff: capacity vs extrapolation
# ---- attention-map viz on a toy sentence ----
words = ['the', 'cat', 'sat', 'on', 'mat'] # toy 5-token sentence
torch.manual_seed(1) # fresh weights: interesting (non-uniform) map
mha2 = MHA(d=16, h=2) # 2 heads: easy to plot side by side
x5 = torch.randn(5, 16) # fake token embeddings for the 5 words
with torch.no_grad(): # no grad: viz only
    _, w5 = mha2(x5, causal=False) # unmasked: every token may look everywhere (clearer map)
fig, ax = plt.subplots(1, 2, figsize=(10, 4)) # two panels: one per head
for hh in range(2): # plot each head
    im = ax[hh].imshow(w5[hh].numpy(), vmin=0, vmax=1) # heatmap: row i = where token i looks
    ax[hh].set_xticks(range(5), words, rotation=30) # x = key tokens
    ax[hh].set_yticks(range(5), words) # y = query tokens
    ax[hh].set_title(f'head {hh} weights') # panel title
fig.colorbar(im, ax=ax.ravel().tolist(), label='weight') # shared colorbar: 0..1
fig.tight_layout() # avoid label overlap
fig.savefig('./results/attn_map.png', dpi=100) # save map for README
print('saved ./results/attn_map.png') # confirm artifact
np.savetxt('./results/attn_weights_h0.csv', w5[0].numpy(), delimiter=',', fmt='%.4f') # dump head-0 weights: inspectable numbers


In [ ]:
# ---- experiments: scale matters / mask matters / heads & length scaling ----
rng = np.random.default_rng(7) # fresh RNG: experiment data independent of earlier cells
dk_big, n_big = 64, 16 # big head dim: saturation shows without 1/sqrt(dk)
Qb = rng.normal(size=(n_big, dk_big)) # unit-magnitude queries (scores std = sqrt(dk) = 8)
Kb = rng.normal(size=(n_big, dk_big)) # unit-magnitude keys (scaled restores std -> 1)
s_raw = Qb @ Kb.T # unscaled scores: variance ~ dk*9 -> softmax saturates
s_sc = s_raw / math.sqrt(dk_big) # scaled scores: variance restored to ~9
e_raw = float(np.abs(softmax_stable(s_raw)).max()) # max weight unscaled: ~1.0 = one-hot, dead grads
e_sc = float(np.abs(softmax_stable(s_sc)).max()) # max weight scaled: spread out, grads flow
print(f'no-scale max-p={e_raw:.4f} vs scaled max-p={e_sc:.4f}') # scale keeps attention soft/trainable
rows = [] # collect what-if rows then write once
rows.append(['unscaled-softmax', 'max_prob', f'{e_raw:.4f}']) # saturation demo row
rows.append(['scaled-softmax', 'max_prob', f'{e_sc:.4f}']) # healthy spread row
Xf = rng.normal(size=(6, 8)) # tiny seq: 6 tokens
W1 = rng.normal(size=(8, 8)) # shared toy projs (same W for fair compare)
o_c, _ = attn_numpy(Xf, W1, W1, W1, mask=causal_mask(6)) # masked output
o_f, _ = attn_numpy(Xf, W1, W1, W1, mask=None) # unmasked output
mdiff = float(np.abs(o_c - o_f).max()) # max change when future becomes visible
rows.append(['causal-vs-full', 'max|diff|', f'{mdiff:.4f}']) # >0: future leaks into past without mask
print(f'causal-vs-full max|diff|={mdiff:.4f} (row0 changes most: alone vs avg-of-future w/o mask)') # leakage quantified
for h in [1, 2, 4, 8]: # sweep head counts at fixed d (flops ~constant)
    torch.manual_seed(SEED) # same init scale each time
    mh = MHA(d=64, h=h) # fresh MHA at this head count
    xx = torch.randn(64, 64) # fixed probe input
    for _ in range(5): # warmup
        mh(xx) # throwaway forward
    t0 = time.perf_counter() # start timer
    for _ in range(20): # timed reps
        mh(xx) # forward only
    ms = (time.perf_counter() - t0) / 20 * 1e3 # mean ms per forward
    rows.append([f'heads={h}', 'ms_per_forward', f'{ms:.3f}']) # one row per head count
print('heads sweep:', ', '.join(f'{r[0]}:{r[2]}ms' for r in rows if r[0].startswith('heads')) + ' (~flat: same flops)') # heads split work, total flops fixed
# length scaling: time the attention kernel alone (QKV precomputed) so O(n^2) is visible
ns, ms_l = [], [] # collect length-scaling data
torch.manual_seed(SEED) # fixed weights: timing only
for nn in [128, 256, 512, 1024]: # seq lengths: 8x range in the quadratic regime (kernel-only, no linears)
    qq = torch.randn(1, 4, nn, 16) # fake Q: 4 heads x dh=16 (precomputed, outside timer)
    kk = torch.randn(1, 4, nn, 16) # fake K
    vv = torch.randn(1, 4, nn, 16) # fake V
    for _ in range(3): # warmup
        F.scaled_dot_product_attention(qq, kk, vv, is_causal=True) # throwaway
    t0 = time.perf_counter() # start
    reps_l = 20 if nn <= 256 else 5 # fewer reps at n=1024 (heavier)
    for _ in range(reps_l): # timed reps
        F.scaled_dot_product_attention(qq, kk, vv, is_causal=True) # kernel only: pure O(n^2 d)
    ms_l.append((time.perf_counter() - t0) / reps_l * 1e3) # mean ms
    ns.append(nn) # store n
coef = float(np.polyfit(np.log(ns), np.log(ms_l), 1)[0]) # log-log slope: 2.0 = quadratic
print(f'length slope(log-log)={coef:.2f} (2.0 means O(n^2))') # expect ~1.5-2 on CPU (matmul+overhead)
rows.append(['length-scaling', 'loglog_slope', f'{coef:.2f}']) # ~2 confirms O(n^2)
fig, ax = plt.subplots(figsize=(6, 4)) # fresh figure for scaling curve
ax.loglog(ns, ms_l, 'o-') # log-log: power law is a straight line
ax.set_xlabel('seq len n') # x label
ax.set_ylabel('ms per causal SDPA kernel') # y label
ax.set_title(f'MHA time vs length (slope={coef:.2f}, quad=2.0)') # title carries the verdict
fig.tight_layout() # avoid clipping
fig.savefig('./results/scaling.png', dpi=100) # save scaling plot
with open('./results/experiments.csv', 'w', newline='') as f: # write what-if table
    w = csv.writer(f) # writer handle
    w.writerow(['experiment', 'metric', 'value']) # header
    w.writerows(rows) # all experiment rows
print('saved ./results/experiments.csv ./results/scaling.png') # confirm artifacts


## What to write in your README (7 questions)

1. What did I build? 2. How does scaled dot-product + causal mask + MHA + positions work?
3. What max|hand-torch|, leak, bench ms, log-log slope? 4. What surprised you?
5. Bottleneck? (hint: O(n²) scores memory/time, fused kernels close the gap).
6. What if no 1/sqrt(d)? no mask? (use `experiments.csv` numbers).
7. Next? (KV-cache in P12, FlashAttention tiling idea from P6).
